# Análise Exploratória de Dados — Telco Customer Churn

**Objetivo:** Descobrir quais são os dados que temos para nosso estudo, utilizando estatística descritiva para sumarizar as principais características do dataset.

---

**Perguntas que iremos responder:**
- Número de exemplares (linhas) e dimensões (colunas)
- Tipos de dados
- Medidas de posição e dispersão
- Distribuição e frequência
- Correlações
- Valores perdidos ou incorretos
- Anomalias e outliers

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": "white"})

df = pd.read_csv("../databese/telco-customer-churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Dataset carregado com sucesso!")
df.head()

## 1. Estrutura do Dataset
Número de exemplares (linhas), dimensões (colunas) e tipos de dados.

In [ ]:
print(f"Shape: {df.shape[0]} linhas x {df.shape[1]} colunas\n")

tipo_df = pd.DataFrame({
    "Tipo": df.dtypes,
    "Não Nulos": df.notnull().sum(),
    "Nulos": df.isnull().sum(),
    "Únicos": df.nunique()
})
tipo_df

## 2. Valores Nulos (Missing Values)
Identificar colunas com dados ausentes e calcular o percentual de missings.

In [ ]:
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(2)
null_df = pd.DataFrame({"Nulos": nulls, "% Missing": nulls_pct})
null_df = null_df[null_df["Nulos"] > 0]

print("Colunas com valores nulos:")
display(null_df)

fig, ax = plt.subplots(figsize=(6, 4))
null_df["Nulos"].plot(kind="bar", ax=ax, color="#e74c3c", edgecolor="white")
ax.set_title("Valores Nulos por Coluna", fontsize=14, fontweight="bold")
ax.set_ylabel("Quantidade")
ax.set_xlabel("")
for p in ax.patches:
    ax.annotate(str(int(p.get_height())),
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=11)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Variável Alvo — Churn
Distribuição da variável que queremos prever.

In [ ]:
churn_counts = df["Churn"].value_counts()
churn_pct = df["Churn"].value_counts(normalize=True) * 100
display(pd.DataFrame({"Contagem": churn_counts, "%": churn_pct.round(2)}))

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

axes[0].pie(
    churn_counts,
    labels=churn_counts.index,
    autopct="%1.1f%%",
    colors=["#2ecc71", "#e74c3c"],
    startangle=90,
    textprops={"fontsize": 13},
    wedgeprops={"edgecolor": "white", "linewidth": 2}
)
axes[0].set_title("Proporção de Churn", fontsize=14, fontweight="bold")

sns.countplot(x="Churn", data=df, ax=axes[1],
              palette={"No": "#2ecc71", "Yes": "#e74c3c"},
              edgecolor="white")
axes[1].set_title("Contagem de Churn", fontsize=14, fontweight="bold")
axes[1].set_xlabel("")
for p in axes[1].patches:
    axes[1].annotate(
        f"{int(p.get_height())}\n({p.get_height()/len(df)*100:.1f}%)",
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha="center", va="bottom", fontsize=12
    )

plt.suptitle("Distribuição da Variável Alvo — Churn", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Medidas de Posição e Dispersão — Variáveis Numéricas

Calculando: Média, Moda, Mediana, Desvio Padrão, Variância, Amplitude, Quartis (Q1, Q3), IQR e Coeficiente de Variação.

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

rows = []
for col in num_cols:
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    rows.append({
        "Coluna":           col,
        "Média":            round(s.mean(), 2),
        "Mediana":          round(s.median(), 2),
        "Moda":             round(s.mode()[0], 2),
        "Desvio Padrão":    round(s.std(), 2),
        "Variância":        round(s.var(), 2),
        "Amplitude":        round(s.max() - s.min(), 2),
        "Q1 (25%)":         round(q1, 2),
        "Q3 (75%)":         round(q3, 2),
        "IQR":              round(iqr, 2),
        "CV (%)":           round((s.std() / s.mean()) * 100, 2),
    })

stats_df = pd.DataFrame(rows).set_index("Coluna")
stats_df.T

## 5. Distribuição das Variáveis Numéricas
Histogramas com KDE e Boxplots separados por Churn.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 15))

for i, col in enumerate(num_cols):
    data_no  = df[df["Churn"] == "No"][col].dropna()
    data_yes = df[df["Churn"] == "Yes"][col].dropna()

    # Histograma + KDE
    ax = axes[i][0]
    ax.hist(data_no,  bins=35, alpha=0.55, color="#2ecc71", label="Não Churn", density=True)
    ax.hist(data_yes, bins=35, alpha=0.55, color="#e74c3c", label="Churn",     density=True)
    data_no.plot.kde(ax=ax,  color="#27ae60", lw=2)
    data_yes.plot.kde(ax=ax, color="#c0392b", lw=2)
    ax.set_title(f"Distribuição — {col}", fontsize=13, fontweight="bold")
    ax.set_xlabel(col)
    ax.set_ylabel("Densidade")
    ax.legend()

    # Boxplot por Churn
    ax2 = axes[i][1]
    sns.boxplot(x="Churn", y=col, data=df, ax=ax2,
                palette={"No": "#2ecc71", "Yes": "#e74c3c"})
    ax2.set_title(f"Boxplot — {col} por Churn", fontsize=13, fontweight="bold")
    ax2.set_xlabel("Churn")
    ax2.set_ylabel(col)

plt.suptitle("Distribuição das Variáveis Numéricas", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 6. Distribuição e Frequência — Variáveis Categóricas
Taxa de Churn (%) por categoria de cada variável.

In [ ]:
cat_cols = [
    "gender", "SeniorCitizen", "Partner", "Dependents",
    "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod",
]

fig, axes = plt.subplots(4, 4, figsize=(22, 20))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ct = (
        df.groupby(col)["Churn"]
        .value_counts(normalize=True)
        .unstack()
        .fillna(0) * 100
    )
    ct.plot(kind="bar", ax=axes[i], stacked=True,
            color=["#2ecc71", "#e74c3c"],
            edgecolor="white", legend=False)
    axes[i].set_title(col, fontsize=12, fontweight="bold")
    axes[i].set_xlabel("")
    axes[i].set_ylabel("%")
    axes[i].tick_params(axis="x", rotation=30)
    axes[i].yaxis.set_major_formatter(mticker.PercentFormatter())

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor="#2ecc71", label="Não Churn"),
                   Patch(facecolor="#e74c3c", label="Churn")]
fig.legend(handles=legend_elements, loc="lower right", fontsize=13, frameon=True)

plt.suptitle("Taxa de Churn (%) por Variável Categórica", fontsize=17, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Correlações
Matriz de correlação entre as variáveis numéricas e a variável alvo (Churn).

In [ ]:
df_corr = df.copy()
df_corr["Churn_num"]      = (df_corr["Churn"] == "Yes").astype(int)
df_corr["SeniorCitizen"]  = df_corr["SeniorCitizen"].astype(int)

corr_cols  = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "Churn_num"]
corr_matrix = df_corr[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
    mask=mask, ax=ax, linewidths=0.5,
    annot_kws={"size": 13}, vmin=-1, vmax=1,
    square=True
)
ax.set_title("Matriz de Correlação", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nCorrelação com Churn:")
display(corr_matrix["Churn_num"].drop("Churn_num").sort_values(ascending=False).to_frame("Correlação com Churn").round(3))

## 8. Anomalias e Outliers
Detecção via método IQR: valores abaixo de `Q1 - 1.5*IQR` ou acima de `Q3 + 1.5*IQR`.

In [ ]:
outlier_rows = []
for col in num_cols:
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((s < lower) | (s > upper)).sum())
    outlier_rows.append({
        "Coluna": col, "Q1": round(q1, 2), "Q3": round(q3, 2),
        "IQR": round(iqr, 2), "Limite Inferior": round(lower, 2),
        "Limite Superior": round(upper, 2), "Outliers": n_out,
        "% Outliers": round(n_out / len(s) * 100, 2)
    })

display(pd.DataFrame(outlier_rows).set_index("Coluna"))

fig, axes = plt.subplots(1, 3, figsize=(14, 6))
for i, col in enumerate(num_cols):
    s = df[col].dropna()
    n_out = outlier_rows[i]["Outliers"]
    pct   = outlier_rows[i]["% Outliers"]
    sns.boxplot(y=s, ax=axes[i], color="#aed6f1",
                flierprops=dict(marker="o", color="#e74c3c", alpha=0.5))
    axes[i].set_title(f"{col}\n{n_out} outliers ({pct}%)", fontsize=12, fontweight="bold")
    axes[i].set_ylabel(col)

plt.suptitle("Detecção de Outliers — Boxplots (IQR)", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Tenure × Churn — Taxa por Tempo de Contrato
Evolução da taxa de churn conforme o tempo que o cliente está na empresa.

In [ ]:
churn_by_tenure = df.groupby("tenure")["Churn"].apply(lambda x: (x == "Yes").mean() * 100)
avg_churn = df["Churn"].eq("Yes").mean() * 100

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(churn_by_tenure.index, churn_by_tenure.values, color="#e74c3c", lw=2, alpha=0.8)
ax.fill_between(churn_by_tenure.index, churn_by_tenure.values, alpha=0.15, color="#e74c3c")
ax.axhline(y=avg_churn, color="#3498db", linestyle="--", lw=1.8,
           label=f"Média geral: {avg_churn:.1f}%")
ax.set_title("Taxa de Churn (%) por Tempo como Cliente (Tenure)", fontsize=14, fontweight="bold")
ax.set_xlabel("Meses como Cliente")
ax.set_ylabel("Taxa de Churn (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 10. MonthlyCharges e TotalCharges × Churn — Violin Plots
Distribuição dos valores de cobrança comparando clientes que saíram ou não.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for i, col in enumerate(["MonthlyCharges", "TotalCharges"]):
    sns.violinplot(x="Churn", y=col, data=df, ax=axes[i],
                   palette={"No": "#2ecc71", "Yes": "#e74c3c"},
                   inner="quartile")
    axes[i].set_title(f"{col} por Churn", fontsize=13, fontweight="bold")
    axes[i].set_xlabel("Churn")

plt.suptitle("Distribuição de Cobranças por Churn — Violin Plots", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

---

## Conclusões da EDA

| # | Insight |
|---|---------|
| 1 | **Dataset**: 7.043 clientes, 21 variáveis. Apenas `TotalCharges` possuía valores nulos (11 registros ≈ 0,16%). |
| 2 | **Desbalanceamento**: ~73,5% dos clientes não cancelaram vs ~26,5% que cancelaram — dado importante para a modelagem. |
| 3 | **Tenure**: Clientes com menos de 12 meses têm taxa de churn muito acima da média. Quanto maior o tempo, menor o churn. |
| 4 | **MonthlyCharges**: Clientes com churn pagam mensalidades mais altas em média. |
| 5 | **TotalCharges**: Clientes que ficam menos tempo e pagam mais mensalidade tendem a sair. |
| 6 | **Contrato**: `Month-to-month` tem taxa de churn drasticamente maior que contratos anuais/bianuais. |
| 7 | **InternetService**: Clientes com fibra óptica têm maior churn do que os com DSL ou sem internet. |
| 8 | **Serviços adicionais**: Ausência de `OnlineSecurity`, `TechSupport` e `OnlineBackup` está associada a maior churn. |
| 9 | **Correlação**: `tenure` tem correlação negativa com churn (-0.35); `MonthlyCharges` tem correlação positiva (+0.19). |
| 10 | **Outliers**: Nenhuma das variáveis numéricas apresentou outliers pelo critério IQR. |